# A temperature sweep through a galaxy

The camera never moves. What moves is the **transfer function**.

A **thin** window in temperature, 0.12 dex wide, which is about ±15 % in temperature, slides from
30 K up to 10⁷ K, and only the gas inside that window is drawn. One phase of the interstellar medium after another lights up in its own
colour and fades again, in the place where it actually lives: the cold gas in a thin flocculent
disc, the warm gas in a thick puffy one, the 10⁵ K interface gas as a web of bubble walls, and the
hot gas as a smooth envelope that fills the whole box.

That last frame is the point of the whole sweep. **The hotter the gas, the smoother and more
volume filling it is.** You can read that off a phase diagram, but seeing where each phase sits in
three dimensions is a different kind of understanding.

The window has to be thin. A wide one adds up everything between its edges, and what arrives is
the sum of several phases at once, which looks like fog whatever you do to it. Narrow it and you
are looking at a surface instead of a slab: at 10⁵·² K the bubble walls of this galaxy resolve
into a web you can trace individual shells through. The second act opens the window past the width
the sweep uses, out to ±1.2 dex and back, so you can see what is lost.

The third act puts **three phases on screen together**, faded in one after another, so you can see
them nested rather than in sequence.

**To run it on your own simulation, change two things in the second cell**: the path and the output
number.

> This is a gallery recipe. It is not part of Mera's test suite, and it is not executed when the
> documentation is built, so its outputs are not stored here. Run it and it will produce them.

**What you need:** a RAMSES output or a converted MERA file with gas. The snapshot used here is
34 million cells after the cut in section 2, about 6 GB of memory, and the 300 frames take
roughly **two and a half hours on eight threads**. Set `PREVIEW = true` in the settings cell to
render three frames instead of three hundred and see the whole design in a few minutes.

**On the pixel size.** 28 pc is chosen, not inherited. The cells that carry the hot gas are 23 to
93 pc, so at 28 pc a cell is about a pixel and the picture is smooth. Render the same thing at
14 pc and the cells become several pixels each, and because a cell contributes one slab of
optical depth to the ray that crosses it, columns of cells along the line of sight show up as
vertical bands. Nothing in the transfer function removes them: a wider band, a slope inside each
cell, trilinear interpolation between cell corners and supersampling were each measured on this
snapshot and none of them touched it, because all four act across the picture while the banding
is along the ray. Removing it needs several samples per cell along the ray, which this renderer
does not do. Until then, **keep the pixel no finer than the cells** and the question does not
arise.

![The temperature window sliding from cold to hot](media/sweep_preview.gif)

*Reduced preview. Full resolution in [`media/sweep_movie.mp4`](media/sweep_movie.mp4).*

| | |
|---|---|
| **Author** | Manuel Behrendt |
| **Contact** | [github.com/ManuelBehrendt](https://github.com/ManuelBehrendt) |
| **Reads** | RAMSES |
| **Provenance** | `Mera v1.8.0 \| L12_SN5_CD_only/output_00390 \| 580.05 Myr \| L=48.0 ndim=3 lmin=6 lmax=12` |
| **Status** | contributed 2026-09-18 |

## The environment

This recipe carries its own `Project.toml`. Running the cell below uses the package versions it was
written against. See
[Reproducibility](https://manuelbehrendt.github.io/Mera.jl/stable/reproducibility/).

In [ ]:
using Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()

## 1. The two lines you change

In [ ]:
using Mera, Printf, Statistics, ColorSchemes, FFMPEG_jll, FileIO
using CairoMakie
CairoMakie.activate!(type = "png")

# ---- change these two ----------------------------------------------------------------
const SIM    = "/path/to/your/simulation"   # a RAMSES output folder, or a MERA-file folder
const OUTPUT = 390                          # the snapshot number
# --------------------------------------------------------------------------------------

const PREVIEW = false        # true renders one frame per act instead of the whole movie
const KEEP_FRAMES = false    # the PNG frames are ~0.5 GB; the movie is the output that matters

const INC, AZ = 65.0, 0.0    # degrees: the camera, fixed for every frame of the movie
const XY, ZH  = 15.0, 12.0   # kpc: half width and half height of the box that is read
const NH_MIN  = 1e-4         # cm^-3: below this there is only smooth halo (see section 2)

const LOGT  = (1.5, 7.2)     # the sweep, in log10 of the temperature in K
const WIDTH = 0.06           # dex: HALF width of the sliding window. Keep it small: this is
                             # what makes the movie show surfaces instead of slabs
const TAU   = 0.5            # the median optical depth each frame is normalised to
const EXPO  = 2.6            # exposure

const PX  = 28.0             # pc, pixel size of a frame. Not finer, deliberately: see below
const FPS = 20
const NSWEEP, NWIDTH, NPHASE = 170, 44, 46     # frames in each of the three acts

## 2. Read the box, and drop the smooth halo

Two decisions, both worth copying.

**The read is a box, not the whole simulation.** 30 × 30 × 24 kpc around the centre holds the disc
and the outflow; the rest of the 48 kpc box is halo that never appears in frame.

**The gas below 10⁻⁴ cm⁻³ is dropped.** That is the smooth circumgalactic medium. It has no
structure to show at this scale, it sits in front of everything along every line of sight, and
because its cells are the coarse ones, each covering many pixels, it costs a large share of the
render time. Removing it takes about a quarter of the cells and **halves the time per frame**,
and the picture gets better rather than worse, because that gas was pure haze.

This is a choice about what to draw, not a claim that the gas is not there. Raise `NH_MIN` and
more of the faint envelope disappears; set it to zero and you keep everything, at twice the
cost.

In [ ]:
gas_all = loaddata(OUTPUT, SIM, :hydro;
                   xrange = [-XY, XY], yrange = [-XY, XY], zrange = [-ZH, ZH],
                   center = [:bc], range_unit = :kpc, verbose = false)

gas = filterdata(gas_all, Above(:rho, NH_MIN, :nH), verbose = false)
gas_all = nothing; GC.gc()          # the full box is not needed again

@printf("%d cells kept above n_H = %.0e cm^-3\n", length(gas.data), NH_MIN)

## 3. What the movie has to normalise, and why

Mera's `:T` is the temperature a fixed mean molecular weight gives, `T = p/ρ · μ m_H/k`, with
μ ≈ 1.32 for neutral primordial gas. In ionised gas the true temperature is about half of it. Use
`:K_mu` if you would rather plot `T/μ` and stay free of the assumption. For a picture whose job is
to separate phases, the proxy is enough, but it is worth saying out loud.

The bigger problem is **abundance**. The census below spans four orders of magnitude in mass per
bin, and a thin window makes that worse, not better: near the peak it still catches a large share
of the gas, out on the hot tail it catches almost nothing. With one fixed opacity for the whole
movie the crowded end is a solid wall and the empty end is invisible, and neither frame tells you
anything.

So each frame is normalised with **`tau_target`** instead of `tau_unit_length`: Mera solves for the
opacity that makes the *median optical depth of that frame* equal to `TAU`, whatever is in the
window. Every phase then arrives at a comparable density on screen.

That normalisation throws information away, so the movie has to give it back. The readout under
each frame carries the **fraction of the gas mass** in the window, which is the quantity the
brightness is no longer allowed to express.

In [ ]:
logT = log10.(getvar(gas, :T, :K))
mass = getvar(gas, :mass, :Msol)
const MTOT = sum(mass)

"Fraction of the gas mass inside a window of half width `w` dex around `10^lt` K."
massfrac(lt, w) = sum(@view mass[abs.(logT .- lt) .<= w]) / MTOT

fig = Figure(size = (900, 330))
ax  = Axis(fig[1, 1], xlabel = "log₁₀ T [K]", ylabel = "gas mass per 0.1 dex [Msol]",
           yscale = log10, title = "where the gas is, and where the window will go")
edges = range(1, 8, step = 0.1)
h = [sum(@view mass[(logT .>= edges[i]) .& (logT .< edges[i+1])]) for i in 1:length(edges)-1]
barplot!(ax, collect(edges[1:end-1]) .+ 0.05, max.(h, 1.0), width = 0.1,
         color = [ColorSchemes.batlow[(e - 1)/7] for e in edges[1:end-1]])
vlines!(ax, [LOGT[1], LOGT[2]], color = :black, linestyle = :dash)
ylims!(ax, 1e3, nothing)
fig

## 4. Where the gas runs out, and what to do about it

Between 10⁴ and 10⁶ K this galaxy is nearly empty, and a thin window there does not show a
surface, it shows **single cells**. The numbers behind that, measured on this snapshot with a
window ±0.06 dex wide:

| centre | cells in the window | share of the gas mass | median AMR level |
|--:|--:|--:|--:|
| 10³·⁵ K | 727 000 | 1.8 % | 12 |
| **10⁴·⁰ K** | **7 814 000** | **13.9 %** | 11 |
| 10⁴·⁵ K | 187 000 | 0.055 % | 11 |
| 10⁵·⁰ K | 189 000 | 0.052 % | 11 |
| 10⁶·⁰ K | 549 000 | 0.045 % | 11 |

Three things pile up at once:

1. **The cooling floor.** Radiative cooling falls off a cliff below 10⁴ K, so gas piles up exactly
   there: one window holds a seventh of all the gas in the galaxy.
2. **Thermal instability just above it.** Gas between 10⁴ and 10⁶ K cools through that range
   quickly, so at any instant almost nothing is sitting in it, 280 times less mass than the window
   next door. What is there is isolated cells caught mid-cooling, not connected sheets.
3. **Those cells are coarser**, level 11 rather than 12, so each is 23 pc against a 28 pc pixel:
   one pixel, with nothing to average it against.

Then `tau_target` finishes the job. It solves for the opacity that puts the median optical depth
at 0.5 *whatever is in the window*, so a band with 280 times less gas gets its opacity raised by
a comparable factor, and every isolated cell is rendered at full strength. The result is stipple.

**Supersampling does not fix this**, which is the useful diagnostic: the dots are real cells, not
pixel aliasing. Widening the window until it holds as much mass as the others does not fix it
either, it only reaches down to the 10⁴ K spike and shows you that instead, which is worse than
stipple because it looks right.

It is worth separating the two things that get confused here. Most published cures for speckle in
a volume rendering treat **sampling**: a ray marched in steps can miss a thin feature, and the
remedies are jitter, more steps, pre-integration or peak finding. None of them apply to
[`volumerender`](@ref), which integrates each cell analytically and composites cells in their
exact front-to-back order, with no step size anywhere. The banding, moiré and wood-grain artefacts
that those methods exist to remove cannot occur here, and that is why supersampling changed
nothing.

What is left is the **data**, and the literature is consistent about the one lever that helps.
Engel, Kraus and Ertl (Graphics Hardware 2001) show that the sampling rate a volume rendering
needs is the product of the Nyquist frequencies of the field and of the transfer function, so
halving the width of a band doubles what it demands, and note that such artefacts "are frequently
observed unless they are avoided by very smooth transfer functions". Knoll et al. (IEEE TVCG 15,
1571, 2009) state the limiting case: an isosurface is a transfer function with a Dirac impulse in
it, and rendering one without artefacts requires "either infinite sampling or smoothing of the
transfer function". Widening the window is not a cosmetic dodge, it is the remedy both papers
name.

What does work is to let the window open across the range that needs it, and **to keep opening it
as the temperature rises**: ±0.06 dex below the cooling floor, ±0.15 by 10⁴·⁴ K, and ±0.25 by
10⁶ K, closing again above it. More is needed at the top of the range than at the bottom, because
the gas there sits in coarser cells as well as being rare, so a thin band catches them one at a
time. Nothing else changes: the opacity target and the exposure stay the same for the whole movie,
so brightness keeps meaning the same thing from the first frame to the last.

**The widths are measured, not inferred.** The obvious rule would be to open the window wherever
the mass in it falls below some threshold, and it is wrong. A window at 10⁶·⁵ K holds 0.048 % of
the mass, as little as one at 10⁴·⁵ K, and it renders perfectly smooth. Sparseness in temperature
is not the criterion; what matters is whether the cells that are there overlap on screen, which
depends on their size as much as on their number. So the widths come from rendering a frame at
each anchor temperature and looking at it, which takes five renders and settles the question.

**One artefact widening does not remove.** In the smooth outer halo the temperature falls slowly
and steadily with radius, so a narrow window picks out a thin *spatial* shell, and on cells of
47 pc that shell breaks into concentric rings, like the contour lines of a map. Widening softens
them, but the cure is to give each cell a footprint by interpolating between cell centres rather
than treating each as a constant block, which this renderer does not do. Raising the density floor
removes the rings by removing the gas that carries them, and costs more than it buys: the picture
breaks into fragments where the threshold cuts through real structure. The rings are left in, and
they are worth recognising for what they are.

In [ ]:
# The width of the window, as a function of where it sits. Every anchor is a temperature where
# a frame was rendered and looked at: WIDTH is thin enough to draw a surface wherever the gas is
# spatially dense, and the empty range needs progressively more, because the gas there is not
# only rare but sits in coarser cells, so a thin band catches them one at a time.
const WIDTH_AT = [(4.0, WIDTH), (4.3, 0.18), (4.7, 0.28), (5.2, 0.35),
                  (5.7, 0.35), (6.0, 0.30), (6.6, WIDTH)]

"The window half width at `lt`, interpolated between the measured anchors and eased so it never
 jumps from one frame to the next."
function window_at(lt)
    (lt <= WIDTH_AT[1][1] || lt >= WIDTH_AT[end][1]) && return WIDTH
    i = findfirst(a -> a[1] >= lt, WIDTH_AT)
    (t0, w0), (t1, w1) = WIDTH_AT[i-1], WIDTH_AT[i]
    f = (lt - t0) / (t1 - t0)
    return w0 + (w1 - w0) * (0.5 - 0.5cos(π * f))
end

for t in (2.9, 4.0, 4.4, 5.2, 6.0, 6.5, 7.0)
    @printf("10^%.1f K: window ±%.2f dex, %.3f %% of the gas mass\n",
            t, window_at(t), 100 * massfrac(t, window_at(t)))
end

## 5. A window, not a threshold

The obvious way to show one temperature would be to move the `colorrange`. It does not work: a
colour range makes everything *above* its top edge opaque as well, so a window at 10³ K still has
the entire hot phase sitting solid in front of it.

What is needed is a **band**, and that is exactly an isosurface. `iso = [T]` with `iso_width` in dex
puts opacity in a shell around `T` and leaves everything else transparent, which is the sliding
window this movie is made of.

The `colorrange` still matters, but for a different job: it is **fixed at 10 K to 10⁸ K for the
whole movie**, so a given temperature keeps the same colour from the first frame to the last. That
is what lets you read the phase off the colour instead of off the caption.

Two more settings earn their place:

* `opacity = (0, 0)` switches the continuous ramp off completely, so the isosurfaces are the only
  thing drawn.
* `supersample = 1` turns off the anti-aliasing pass. It **halves the time per frame**, and on a
  moving picture nobody sees the difference. Leave it at `:auto` for a still.

In [ ]:
const VIEW = (center = [:bc], range_unit = :kpc,
              xrange = [-XY, XY], yrange = [-XY, XY], zrange = [-ZH, ZH],
              inclination = INC, azimuth = AZ, axis = :angmom,
              pxsize = [PX, :pc], verbose = false)

# colour fixed over the whole movie: 10 K at one end, 10^8 K at the other
const LOOK = (unit = :K, colorrange = (1e1, 1e8), colormap = :batlow, opacity = (0.0, 0.0))

render(; kw...) = volumerender(gas, :T; LOOK..., supersample = 1, kw..., VIEW...)

m0 = render(iso = [10^5.2], iso_width = WIDTH, iso_alpha = 0.7, tau_target = TAU, exposure = EXPO)
const AXW, AXH = size(m0.image)
@printf("frame %d x %d px\n", AXW, AXH)

## 6. The readout is the transfer function itself

Under every frame sits the colour bar the render is using, from 10 K to 10⁸ K, with the window
outlined on it in white. In the first act the outline slides; in the second it opens and closes; in
the third there are three of them. So the viewer watches the transfer function and its result at
the same time, which is the only honest way to show a picture whose content is a choice.

**The axis is pinned in pixels.** Left to itself, Makie sizes the axis from its labels and the
result drifts by a pixel or two between frames, which in a movie reads as a jitter. Fixing `width`
and `height` makes every frame identical in geometry.

In [ ]:
const BAR  = reshape([ColorSchemes.batlow[x] for x in range(0, 1, length = 512)], :, 1)
const TICKS = (1:8, ["10¹", "10²", "10³", "10⁴", "10⁵", "10⁶", "10⁷", "10⁸"])

"The phase a temperature belongs to. A label for the eye, not a definition."
phase(lt) = lt < 3   ? "cold neutral" :
            lt < 4   ? "warm neutral" :
            lt < 5.5 ? "warm ionised" :
            lt < 6.5 ? "interfaces and bubble walls" : "hot outflow"

function frame!(path, m, windows, line1, line2)
    fig = Figure(size = (AXW + 40, AXH + 150), backgroundcolor = :black,
                 figure_padding = (20, 20, 10, 14))

    ax = Axis(fig[1, 1]; width = AXW, height = AXH, backgroundcolor = :black)
    hidedecorations!(ax); hidespines!(ax)
    x0, x1, y0, y1 = getextent(m, :kpc)
    image!(ax, x0 .. x1, y0 .. y1, m.image, interpolate = false)

    Label(fig[2, 1], line1, color = :white, fontsize = 30, halign = :left, tellwidth = false,
          padding = (24, 0, 10, 2))
    Label(fig[3, 1], line2, color = (:white, 0.65), fontsize = 20, halign = :left,
          tellwidth = false, padding = (24, 0, 0, 6))

    axb = Axis(fig[4, 1]; width = AXW - 8, height = 22, backgroundcolor = :black,
               xticks = TICKS, xticklabelcolor = :white, xticklabelsize = 17,
               bottomspinecolor = :white, topspinecolor = :white,
               leftspinecolor = :white, rightspinecolor = :white,
               xtickcolor = :white, xgridvisible = false, ygridvisible = false)
    hideydecorations!(axb)
    image!(axb, 1 .. 8, 0 .. 1, BAR)
    for (lo, hi) in windows                       # the window, drawn on the scale it acts on
        poly!(axb, Rect(lo, 0.0, hi - lo, 1.0), color = (:white, 0.0),
              strokecolor = :white, strokewidth = 3)
    end
    xlims!(axb, 1, 8); ylims!(axb, 0, 1)

    # the layout decides the height; px_per_unit = 1 keeps one output pixel per rendered pixel,
    # so the picture is never resampled and every frame comes out the same size
    resize_to_layout!(fig)
    save(path, fig; px_per_unit = 1)
    return nothing
end

## 7. The three acts

Each act is a list of frames, built before anything is rendered, so the whole movie can be checked
as a table of settings before two hours are spent on it.

**Act 1, the sweep.** One window, sliding from 10¹·⁵ to 10⁷·² K, opening as it crosses the empty
range between 10⁴ and 10⁶ K and closing again after it. The width is on screen in every frame, so
you can watch it breathe from ±0.06 to ±0.15 dex and back.

**Act 2, the width.** The window stops at 10⁵·² K, the temperature where this galaxy has its
bubble walls, and opens from the width the sweep itself uses there, ±0.35 dex, out to ±1.2 dex and
back. It starts there rather than at nothing on purpose: narrower than the sweep's own width, the
speckle of section 4 comes back, and this act is about the other end of the trade. A band at the
chosen width draws the surface where gas crosses that temperature and the shells of individual
bubbles stay separate; a wide one adds up everything between its edges, the shells merge into
their own sum, and no setting downstream recovers them.

**Act 3, the phases together.** Three windows, at 10²·⁵, 10⁵·² and 10⁶·⁵ K, faded in one after the
other so you see the cold disc first, then the bubble walls around it, then the hot envelope over
both. Each surface takes **its own width** from the same rule, ±0.06 dex for the cold one and
±0.35 for the middle, which is what `iso_width` accepting one value per surface is for. A single
width cannot serve all three: wide enough for the middle one turns the cold disc into a slab,
thin enough for the cold one brings the middle one's speckle back.

Act 3 is the one act that **cannot use `tau_target`**: renormalising every frame would cancel the
fade exactly as it was drawn. It uses a fixed `tau_unit_length` instead, which is also why its
three surfaces have hand-picked opacities, the widest phase getting the smallest.

In [ ]:
const HOLD = 8                       # frames held at the end of an act, so the eye can settle

# The window moves at a constant speed in log T, and the step is smaller than its half width, so
# consecutive frames overlap. A step larger than the window would make the movie strobe: gas would
# jump from one frame to the next instead of moving through the window.
const STEP = (LOGT[2] - LOGT[1]) / (NSWEEP - 1)
STEP < WIDTH || @warn "the window steps further than it is wide; raise NSWEEP or WIDTH" STEP WIDTH

plan = NamedTuple[]

# --- act 1: the window slides from cold to hot --------------------------------------------
for k in 0:(NSWEEP - 1)
    lt = LOGT[1] + k * STEP
    w  = window_at(lt)
    mf = massfrac(lt, w)
    push!(plan, (iso = [10.0^lt], width = w, alpha = [0.7], target = TAU, tau = nothing,
                 expo = EXPO, windows = [(lt - w, lt + w)],
                 line1 = @sprintf("T = 10^%.2f K, window ±%.2f dex", lt, w),
                 line2 = @sprintf("%s  ·  %.3f %% of the gas mass", phase(lt), 100mf)))
end
append!(plan, fill(plan[end], HOLD))

# --- act 2: the same temperature, the window opening and closing ---------------------------
const LT2 = 5.2
const W2  = window_at(LT2)        # start from the width the sweep itself uses here
for k in 0:(NWIDTH - 1)
    w  = W2 + (1.2 - W2) * (0.5 - 0.5cos(2π * k / NWIDTH))
    mf = massfrac(LT2, w)
    push!(plan, (iso = [10.0^LT2], width = w, alpha = [0.7], target = TAU, tau = nothing,
                 expo = EXPO, windows = [(LT2 - w, LT2 + w)],
                 line1 = @sprintf("T = 10^%.1f K, window ±%.2f dex", LT2, w),
                 line2 = @sprintf("a thin shell, or everything between  ·  %.3f %% of the gas mass",
                                  100mf)))
end
append!(plan, fill(plan[end], HOLD))

# --- act 3: three phases, faded in one after another ---------------------------------------
const TRI  = [2.5, 5.2, 6.5]          # log10 K
const WTRI = window_at.(TRI)          # each surface gets the width its own temperature needs
# The hot phase fills the whole box, so a ray crosses 20 kpc of it and a tenth of the opacity of
# the cold filaments would still bury them. Hence 0.0025 against 1.0: the number that matters is
# opacity times path length, not opacity. The middle surface is wide, so it too needs a small
# value or it hides the cold disc it is supposed to sit around.
const ATRI = [1.0, 0.03, 0.0025]
for k in 0:(NPHASE - 1)
    f = k / (NPHASE - 1)
    a = [clamp(3f - (i - 1), 0, 1) * ATRI[i] for i in 1:3]
    on = findall(>(1e-5), a)
    push!(plan, (iso = 10.0 .^ TRI, width = WTRI, alpha = max.(a, 1e-6),
                 target = nothing, tau = [600, :pc], expo = 10.0,
                 windows = [(TRI[i] - WTRI[i], TRI[i] + WTRI[i]) for i in on],
                 line1 = "three phases at once",
                 # never empty: a label with no text collapses its row and that one frame comes
                 # out a few pixels shorter than the rest, which the encoder rejects
                 line2 = isempty(on) ? "fading in" :
                         join([@sprintf("10^%.1f K", TRI[i]) for i in on], "   +   ")))
end
append!(plan, fill(plan[end], 3HOLD))

@printf("%d frames: %d sweep + %d width + %d phases (plus holds)\n",
        length(plan), NSWEEP, NWIDTH, NPHASE)

## 8. Render

`tau_target` costs an extra pass over the cells, so a frame that uses it is slower than one that
does not. The progress line prints the running average, which is the number to watch if you want
to know when it will finish.

In [ ]:
frames = joinpath("media", "sweep_frames"); mkpath(frames)
todo = PREVIEW ? [plan[NSWEEP ÷ 3], plan[NSWEEP + NWIDTH ÷ 2], plan[end]] : plan

t0 = time()
for (k, p) in enumerate(todo)
    kw = p.target === nothing ? (tau_unit_length = p.tau,) : (tau_target = p.target,)
    m  = render(; iso = p.iso, iso_width = p.width, iso_alpha = p.alpha, exposure = p.expo, kw...)
    frame!(joinpath(frames, @sprintf("f_%04d.png", k - 1)), m, p.windows, p.line1, p.line2)
    el = time() - t0
    @printf("\rframe %3d/%3d  %.0f s each  %.0f min left        ",
            k, length(todo), el / k, (el / k) * (length(todo) - k) / 60)
end
println()

## 9. Assemble

The encoder is **`FFMPEG_jll`**, the ffmpeg that Julia's package manager installs, rather than
whatever `ffmpeg` happens to be on your path. It is one line in `Project.toml` and it removes the
one step of this recipe that a reader could not reproduce. It is also not a theoretical worry: the
ffmpeg that came with this machine's Python distribution aborts on a missing library, and the
recipe would have failed at the last cell with nothing rendered wrong.

The mp4 is the output. The gif is only a preview small enough to sit inside a page, so it is cut
to 12 frames a second and 560 pixels wide.

In [ ]:
# every frame must have the same pixel size or the encoder refuses them
sizes = unique([size(FileIO.load(joinpath(frames, f))) for f in sort(readdir(frames))
                if endswith(f, ".png")])
length(sizes) == 1 || error("frames differ in size: $sizes")

pat = joinpath(frames, "f_%04d.png")
media(x) = joinpath("media", x)
pal = joinpath(tempdir(), "sweep_palette.png")

ffmpeg() do exe
    # crop, not scale: the frames keep their own resolution, and the crop only trims an odd
    # row or column, which libx264 requires
    run(`$exe -y -loglevel error -framerate $FPS -i $pat
         -vf "crop=trunc(iw/2)*2:trunc(ih/2)*2" -c:v libx264 -pix_fmt yuv420p -crf 20
         -movflags +faststart $(media("sweep_movie.mp4"))`)
    run(`$exe -y -loglevel error -i $pat
         -vf "fps=12,scale=560:-2:flags=lanczos,palettegen=max_colors=128" $pal`)
    run(`$exe -y -loglevel error -framerate $FPS -i $pat -i $pal
         -lavfi "fps=12,scale=560:-2:flags=lanczos[x];[x][1:v]paletteuse=dither=none"
         -loop 0 $(media("sweep_preview.gif"))`)
end

@printf("%d frames of %s  ->  mp4 %.1f MB, gif %.1f MB\n",
        length(readdir(frames)), string(sizes[1]),
        filesize(media("sweep_movie.mp4"))/1e6, filesize(media("sweep_preview.gif"))/1e6)

# At this resolution the frames are about half a gigabyte, which is not worth keeping once the
# movie exists. Set KEEP_FRAMES = true above if you want to re-encode without re-rendering.
KEEP_FRAMES || rm(frames; recursive = true)

## The thumbnail

The card wants one picture that says what the recipe makes. A single frame from the sweep says
"a galaxy"; the four phases side by side say "a sweep". This renders them at full quality, with
`supersample` back on, since it is a still and the extra pass is affordable once.

In [ ]:
fig = Figure(size = (1720, 470), backgroundcolor = :black, figure_padding = 6)
for (k, lt) in enumerate((2.5, 4.0, 5.2, 6.5))
    m = volumerender(gas, :T; LOOK..., iso = [10.0^lt], iso_width = window_at(lt), iso_alpha = 0.7,
                     tau_target = TAU, exposure = EXPO, VIEW...)
    ax = Axis(fig[1, k]; backgroundcolor = :black,
              title = @sprintf("10^%.1f K", lt), titlecolor = :white, titlesize = 24)
    hidedecorations!(ax); hidespines!(ax)
    x0, x1, y0, y1 = getextent(m, :kpc)
    image!(ax, x0 .. x1, y0 .. y1, m.image)
end
save(media("sweep_phases.png"), fig)
makethumb(media("sweep_phases.png"), media("sweep_thumb.png"))
fig

## Attribution: run this once and paste the result

In [ ]:
println("Reads        ", gas.info.simcode)
println("Mera         ", mera_build())
println("Provenance   ", provenance_string(gas))

## Making it yours

- **A different quantity.** Nothing here is about temperature. Sweep a window through metallicity
  and you watch enriched gas leave the disc; through density and you peel the galaxy apart shell by
  shell; through `:vlos`, with a symmetric range and a diverging colour map, and the two halves of
  the rotation separate. Change `:T` and the colour range, and the rest of the recipe stands.
- **A moving camera as well.** `volumeorbit` turns the view with one fixed look. Combining both,
  a turning camera and a sliding window, usually shows less rather than more, because the eye
  cannot tell which of the two caused a change. If you want both, do them in separate acts.
- **A different band shape.** `iso_width` gives a smooth bump. For a hard-edged slab, select the
  cells instead, with `filterdata(gas, InRange(:T, lo, hi, :K))`, and render those with a plain
  opacity ramp. The picture is crisper and the frames are much faster, at the cost of a visible
  step as the window moves.
- **Numbers instead of pictures.** The same loop with `msum` over each window, rather than a
  render, is the mass-temperature distribution the movie is standing in for. Plotting the two side
  by side is a good way to check that the picture is not lying to you.
- **The cheap version.** `PREVIEW = true` renders one frame per act. Raise `PX` to 100 pc and drop
  the frame counts to a fifth to get the whole movie in ten minutes while you tune the look.
- **A continuous field.** `interpolate=true` puts the values on the cell corners and interpolates
  trilinearly inside, instead of treating each cell as one flat block. It costs a few per cent and
  is worth having when cells are large in the frame. It is off here because at 28 pc the cells are
  already about a pixel, so there is nothing for it to smooth.